# 03 - BERT Frozen Embeddings
**Author:** Sanjeev Kumar | IIT Bombay

Test frozen DistilBERT CLS and mean pooling before fine-tuning.

In [ ]:
import pandas as pd, numpy as np, re, os, sys, warnings, torch
import joblib
warnings.filterwarnings('ignore')
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load data (same as notebook 02)
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
PRODUCT_MAP = {
    'Credit reporting or other personal consumer reports': 'Credit Reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit Reporting',
    'Credit reporting': 'Credit Reporting', 'Debt collection': 'Debt Collection',
    'Mortgage': 'Mortgage', 'Checking or savings account': 'Bank Account',
    'Bank account or service': 'Bank Account', 'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card', 'Prepaid card': 'Credit Card',
    'Student loan': 'Loans', 'Vehicle loan or lease': 'Loans', 'Consumer Loan': 'Loans',
    'Payday loan, title loan, personal loan, or advance loan': 'Loans',
    'Payday loan, title loan, or personal loan': 'Loans', 'Payday loan': 'Loans',
    'Money transfer, virtual currency, or money service': 'Money Transfer',
    'Money transfers': 'Money Transfer', 'Debt or credit management': 'Debt Collection',
    'Other financial service': 'Other',
}
def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'x{2,}', 'XXXX', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv(url, compression='zip', low_memory=False, nrows=200000)
df_text = df[df['Consumer complaint narrative'].notna()].copy()
df_text['product_clean'] = df_text['Product'].map(PRODUCT_MAP).fillna('Other')
df_text['narrative_clean'] = df_text['Consumer complaint narrative'].apply(clean_text)
df_text['Date received'] = pd.to_datetime(df_text['Date received'])
df_sorted = df_text.sort_values('Date received').reset_index(drop=True)
n = len(df_sorted)
train_df = df_sorted.iloc[:int(n*0.70)]
val_df   = df_sorted.iloc[int(n*0.70):int(n*0.85)]
test_df  = df_sorted.iloc[int(n*0.85):]
X_train = train_df['narrative_clean'].values
X_val   = val_df['narrative_clean'].values
X_test  = test_df['narrative_clean'].values
y_train_prod = train_df['product_clean'].values
y_val_prod   = val_df['product_clean'].values
y_test_prod  = test_df['product_clean'].values
print(f"Train:{len(train_df):,} Val:{len(val_df):,} Test:{len(test_df):,}")

In [ ]:
# Load DistilBERT
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)
print(f"Model loaded on: {device}")

def get_embeddings(texts, batch_size=128, max_length=256, pooling='cls'):
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = bert_model(**enc)
        if pooling == 'cls':
            emb = out.last_hidden_state[:,0,:]
        else:
            mask = enc['attention_mask'].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_emb.append(emb.cpu().numpy())
        if (i // batch_size) % 20 == 0:
            print(f"  {min(i+batch_size,len(texts)):,}/{len(texts):,}")
    return np.vstack(all_emb)

print("\nGenerating CLS embeddings...")
X_train_cls = get_embeddings(X_train, pooling='cls')
X_val_cls   = get_embeddings(X_val,   pooling='cls')
X_test_cls  = get_embeddings(X_test,  pooling='cls')
print("\nGenerating Mean pooling embeddings...")
X_train_mean = get_embeddings(X_train, pooling='mean')
X_val_mean   = get_embeddings(X_val,   pooling='mean')
X_test_mean  = get_embeddings(X_test,  pooling='mean')
np.save('../models/X_train_cls.npy',  X_train_cls)
np.save('../models/X_val_cls.npy',    X_val_cls)
np.save('../models/X_test_cls.npy',   X_test_cls)
np.save('../models/X_train_mean.npy', X_train_mean)
np.save('../models/X_val_mean.npy',   X_val_mean)
np.save('../models/X_test_mean.npy',  X_test_mean)
print(f"Shapes: CLS={X_train_cls.shape} Mean={X_train_mean.shape}")

In [ ]:
# Train classifiers on frozen embeddings
le = LabelEncoder()
le.fit(y_train_prod)

results = {}
for name, Xtr, Xva, Xte in [
    ('CLS + LR',   X_train_cls,  X_val_cls,  X_test_cls),
    ('Mean + LR',  X_train_mean, X_val_mean, X_test_mean),
]:
    clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42, n_jobs=-1)
    clf.fit(Xtr, y_train_prod)
    vf1 = f1_score(y_val_prod,  clf.predict(Xva), average='macro')
    tf1 = f1_score(y_test_prod, clf.predict(Xte), average='macro')
    print(f"BERT {name}: Val={vf1:.4f} Test={tf1:.4f}")
    results[f'BERT {name}'] = {'val': vf1, 'test': tf1}

# XGBoost on mean pooling
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=42, verbosity=0)
xgb.fit(X_train_mean, le.transform(y_train_prod), eval_set=[(X_val_mean, le.transform(y_val_prod))], verbose=False)
vf1 = f1_score(y_val_prod,  le.inverse_transform(xgb.predict(X_val_mean)),  average='macro')
tf1 = f1_score(y_test_prod, le.inverse_transform(xgb.predict(X_test_mean)), average='macro')
print(f"BERT Mean + XGB: Val={vf1:.4f} Test={tf1:.4f}")
results['BERT Mean + XGB'] = {'val': vf1, 'test': tf1}

print("\nFinding: Frozen BERT embeddings WORSE than TF-IDF!")
print("Reason: BERT not adapted to CFPB complaint language yet!")
print("Solution: Fine-tune DistilBERT in next notebook!")

## Finding

Frozen BERT embeddings **underperform TF-IDF** for keyword-heavy product classification.

> This is expected! BERT needs fine-tuning to learn domain-specific patterns.

**Next → 04_bert_finetuning.ipynb**